In [0]:
path = "/Volumes/workspace/default/nycyellowtaxi/yellow_tripdata_2025-01.parquet"
df = spark.read.parquet(path)
display(df.limit(10))
print(df.count())

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
1,2025-01-01T00:18:38.000,2025-01-01T00:26:59.000,1,1.6,1,N,229,237,1,10.0,3.5,0.5,3.0,0.0,1.0,18.0,2.5,0.0,0.0
1,2025-01-01T00:32:40.000,2025-01-01T00:35:13.000,1,0.5,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
1,2025-01-01T00:44:04.000,2025-01-01T00:46:01.000,1,0.6,1,N,141,141,1,5.1,3.5,0.5,2.0,0.0,1.0,12.1,2.5,0.0,0.0
2,2025-01-01T00:14:27.000,2025-01-01T00:20:01.000,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.0,0.0,1.0,9.7,0.0,0.0,0.0
2,2025-01-01T00:21:34.000,2025-01-01T00:25:06.000,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.0,0.0,1.0,8.3,0.0,0.0,0.0
2,2025-01-01T00:48:24.000,2025-01-01T01:08:26.000,2,2.63,1,N,239,68,2,19.1,1.0,0.5,0.0,0.0,1.0,24.1,2.5,0.0,0.0
1,2025-01-01T00:14:47.000,2025-01-01T00:16:15.000,0,0.4,1,N,170,170,1,4.4,3.5,0.5,2.35,0.0,1.0,11.75,2.5,0.0,0.0
1,2025-01-01T00:39:27.000,2025-01-01T00:51:51.000,0,1.6,1,N,234,148,1,12.1,3.5,0.5,2.0,0.0,1.0,19.1,2.5,0.0,0.0
1,2025-01-01T00:53:43.000,2025-01-01T01:13:23.000,0,2.8,1,N,148,170,1,19.1,3.5,0.5,3.0,0.0,1.0,27.1,2.5,0.0,0.0
2,2025-01-01T00:00:02.000,2025-01-01T00:09:36.000,1,1.71,1,N,237,262,2,11.4,1.0,0.5,0.0,0.0,1.0,16.4,2.5,0.0,0.0


3475226


In [0]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



Silver (Cleaning and Quality)

In [0]:
from pyspark.sql.functions import col, when, lit

df = df.withColumn('is_valid',
    when(
        (col('trip_distance') >= 0) &
        (col('fare_amount') >= 0) &
        (col('passenger_count').between(0, 6)) &
        (col('tpep_pickup_datetime') < col('tpep_dropoff_datetime')),
        True
    ).otherwise(False)
)

df = df.withColumn('invalid_reason',
    when((col('trip_distance') < 0), lit("trip_distance_negative"))
    .when((col('fare_amount') < 0), lit("fare_amount_negative"))
    .when((col('passenger_count') < 0) | (col('passenger_count') > 6), lit("passenger_out_of_range"))
    .when((col('tpep_pickup_datetime') > col('tpep_dropoff_datetime')), lit("pickup_after_dropoff"))
)

display(df.limit(10))

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,is_valid,invalid_reason
1,2025-01-01T00:18:38.000,2025-01-01T00:26:59.000,1,1.6,1,N,229,237,1,10.0,3.5,0.5,3.0,0.0,1.0,18.0,2.5,0.0,0.0,true,null
1,2025-01-01T00:32:40.000,2025-01-01T00:35:13.000,1,0.5,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,true,null
1,2025-01-01T00:44:04.000,2025-01-01T00:46:01.000,1,0.6,1,N,141,141,1,5.1,3.5,0.5,2.0,0.0,1.0,12.1,2.5,0.0,0.0,true,null
2,2025-01-01T00:14:27.000,2025-01-01T00:20:01.000,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.0,0.0,1.0,9.7,0.0,0.0,0.0,true,null
2,2025-01-01T00:21:34.000,2025-01-01T00:25:06.000,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.0,0.0,1.0,8.3,0.0,0.0,0.0,true,null
2,2025-01-01T00:48:24.000,2025-01-01T01:08:26.000,2,2.63,1,N,239,68,2,19.1,1.0,0.5,0.0,0.0,1.0,24.1,2.5,0.0,0.0,true,null
1,2025-01-01T00:14:47.000,2025-01-01T00:16:15.000,0,0.4,1,N,170,170,1,4.4,3.5,0.5,2.35,0.0,1.0,11.75,2.5,0.0,0.0,true,null
1,2025-01-01T00:39:27.000,2025-01-01T00:51:51.000,0,1.6,1,N,234,148,1,12.1,3.5,0.5,2.0,0.0,1.0,19.1,2.5,0.0,0.0,true,null
1,2025-01-01T00:53:43.000,2025-01-01T01:13:23.000,0,2.8,1,N,148,170,1,19.1,3.5,0.5,3.0,0.0,1.0,27.1,2.5,0.0,0.0,true,null
2,2025-01-01T00:00:02.000,2025-01-01T00:09:36.000,1,1.71,1,N,237,262,2,11.4,1.0,0.5,0.0,0.0,1.0,16.4,2.5,0.0,0.0,true,null


In [0]:
silver_nyc_yellow_taxi_bad_records = df.filter(df['is_valid'] == False)
silver_nyc_yellow_taxi = df.filter(df['is_valid'] == True)

print("all count",df.count())
print("bad record",silver_nyc_yellow_taxi_bad_records.count())
display(silver_nyc_yellow_taxi_bad_records.limit(10))
print("good records",silver_nyc_yellow_taxi.count())
display(silver_nyc_yellow_taxi.limit(10))

all count 3475226
bad record 601293


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,is_valid,invalid_reason
2,2025-01-01T00:01:41.000,2025-01-01T00:07:14.000,1,0.71,1,N,79,107,2,-7.2,-1.0,-0.5,3.66,0.0,-1.0,-8.54,-2.5,0.0,0.0,false,fare_amount_negative
2,2025-01-01T00:55:54.000,2025-01-01T01:00:38.000,1,0.69,1,N,137,233,4,-6.5,-1.0,-0.5,0.0,0.0,-1.0,-11.5,-2.5,0.0,0.0,false,fare_amount_negative
1,2025-01-01T00:49:48.000,2025-01-01T00:49:48.000,1,0.0,1,Y,87,264,2,20.06,0.0,0.0,0.0,0.0,0.0,20.06,0.0,0.0,0.0,false,null
2,2025-01-01T00:04:29.000,2025-01-01T00:55:58.000,9,31.97,5,N,132,265,2,90.0,0.0,0.0,0.0,20.32,1.0,111.32,0.0,0.0,0.0,false,passenger_out_of_range
2,2025-01-01T00:56:12.000,2025-01-01T01:15:00.000,1,0.97,1,N,161,170,4,-16.3,-1.0,-0.5,0.0,0.0,-1.0,-21.3,-2.5,0.0,0.0,false,fare_amount_negative
2,2025-01-01T00:55:53.000,2025-01-01T01:06:49.000,1,1.42,1,N,79,45,2,-12.1,-1.0,-0.5,0.0,0.0,-1.0,-17.1,-2.5,0.0,0.0,false,fare_amount_negative
2,2025-01-01T00:29:35.000,2025-01-01T00:36:02.000,1,0.6,1,N,79,148,4,-7.2,-1.0,-0.5,0.0,0.0,-1.0,-12.2,-2.5,0.0,0.0,false,fare_amount_negative
2,2025-01-01T00:11:44.000,2025-01-01T00:25:41.000,2,1.88,1,N,79,161,4,-14.2,-1.0,-0.5,0.0,0.0,-1.0,-19.2,-2.5,0.0,0.0,false,fare_amount_negative
2,2025-01-01T00:11:58.000,2025-01-01T00:12:31.000,1,0.01,1,N,42,42,4,-3.0,-1.0,-0.5,0.0,0.0,-1.0,-5.5,0.0,0.0,0.0,false,fare_amount_negative
2,2025-01-01T00:09:58.000,2025-01-01T00:14:28.000,1,0.6,1,N,140,263,2,-6.5,-1.0,-0.5,0.0,0.0,-1.0,-11.5,-2.5,0.0,0.0,false,fare_amount_negative


good records 2873933


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,is_valid,invalid_reason
1,2025-01-01T00:18:38.000,2025-01-01T00:26:59.000,1,1.6,1,N,229,237,1,10.0,3.5,0.5,3.0,0.0,1.0,18.0,2.5,0.0,0.0,true,null
1,2025-01-01T00:32:40.000,2025-01-01T00:35:13.000,1,0.5,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,true,null
1,2025-01-01T00:44:04.000,2025-01-01T00:46:01.000,1,0.6,1,N,141,141,1,5.1,3.5,0.5,2.0,0.0,1.0,12.1,2.5,0.0,0.0,true,null
2,2025-01-01T00:14:27.000,2025-01-01T00:20:01.000,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.0,0.0,1.0,9.7,0.0,0.0,0.0,true,null
2,2025-01-01T00:21:34.000,2025-01-01T00:25:06.000,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.0,0.0,1.0,8.3,0.0,0.0,0.0,true,null
2,2025-01-01T00:48:24.000,2025-01-01T01:08:26.000,2,2.63,1,N,239,68,2,19.1,1.0,0.5,0.0,0.0,1.0,24.1,2.5,0.0,0.0,true,null
1,2025-01-01T00:14:47.000,2025-01-01T00:16:15.000,0,0.4,1,N,170,170,1,4.4,3.5,0.5,2.35,0.0,1.0,11.75,2.5,0.0,0.0,true,null
1,2025-01-01T00:39:27.000,2025-01-01T00:51:51.000,0,1.6,1,N,234,148,1,12.1,3.5,0.5,2.0,0.0,1.0,19.1,2.5,0.0,0.0,true,null
1,2025-01-01T00:53:43.000,2025-01-01T01:13:23.000,0,2.8,1,N,148,170,1,19.1,3.5,0.5,3.0,0.0,1.0,27.1,2.5,0.0,0.0,true,null
2,2025-01-01T00:00:02.000,2025-01-01T00:09:36.000,1,1.71,1,N,237,262,2,11.4,1.0,0.5,0.0,0.0,1.0,16.4,2.5,0.0,0.0,true,null


# Gold (Analytics-ready)

In [0]:
from pyspark.sql import functions as F

gold_base = (
    silver_nyc_yellow_taxi
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_day_of_week", F.dayofweek("tpep_pickup_datetime"))
    .withColumn(
        "trip_duration_minutes",
        (
            F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")
        ) / 60.0
    )
    .withColumn("revenue", F.col("total_amount"))
)

display(gold_base.select(
    "pickup_date","pickup_hour","pickup_day_of_week",
    "trip_duration_minutes","trip_distance","fare_amount","revenue",
    "PULocationID","DOLocationID","payment_type"
).limit(10))

pickup_date,pickup_hour,pickup_day_of_week,trip_duration_minutes,trip_distance,fare_amount,revenue,PULocationID,DOLocationID,payment_type
2025-01-01,0,4,8.35,1.6,10.0,18.0,229,237,1
2025-01-01,0,4,2.55,0.5,5.1,12.12,236,237,1
2025-01-01,0,4,1.95,0.6,5.1,12.1,141,141,1
2025-01-01,0,4,5.566666666666666,0.52,7.2,9.7,244,244,2
2025-01-01,0,4,3.533333333333333,0.66,5.8,8.3,244,116,2
2025-01-01,0,4,20.033333333333335,2.63,19.1,24.1,239,68,2
2025-01-01,0,4,1.4666666666666666,0.4,4.4,11.75,170,170,1
2025-01-01,0,4,12.4,1.6,12.1,19.1,234,148,1
2025-01-01,0,4,19.666666666666668,2.8,19.1,27.1,148,170,1
2025-01-01,0,4,9.566666666666666,1.71,11.4,16.4,237,262,2


### 1) gold_daily_kpis
Trips per day + averages + total revenue (based on pickup_date).

In [0]:
gold_daily_kpis = (
    gold_base
    .groupBy("pickup_date")
    .agg(
        F.count("*").alias("trips"),
        F.avg("trip_distance").alias("avg_distance"),
        F.avg("fare_amount").alias("avg_fare"),
        F.sum("revenue").alias("total_revenue"),
    )
    .orderBy("pickup_date")
)

display(gold_daily_kpis)

pickup_date,trips,avg_distance,avg_fare,total_revenue
2024-12-31,21,3.6576190476190478,19.123809523809523,589.1699999999998
2025-01-01,72083,3.8715559840738742,21.02977054229181,2147244.439999875
2025-01-02,79231,3.681581325491248,20.521874392598924,2358524.459999876
2025-01-03,85515,3.3743469566742705,19.26436566684211,2417050.699999952
2025-01-04,90987,3.2385835339114224,18.663046369261895,2448197.149999915
2025-01-05,73927,3.830293532809353,20.27586321641634,2192343.989999924
2025-01-06,74516,3.5289709592570753,19.254641419292682,2161377.6299999487
2025-01-07,90475,3.063036308372504,17.69132312793612,2460592.9999999474
2025-01-08,97862,2.9145598904580075,17.16831006928137,2608901.31999997
2025-01-09,101971,3.0296150866422944,17.727839385707867,2788925.1799999387


### 2) gold_zone_heatmap
Trips per pickup/dropoff zone pair. (Optionally enrich with zone names using `taxi_zone_lookup.csv`.)

In [0]:
# Base heatmap (IDs)
gold_zone_heatmap = (
    gold_base
    .groupBy("PULocationID", "DOLocationID")
    .agg(F.count("*").alias("trips"))
    .orderBy(F.col("trips").desc())
)

display(gold_zone_heatmap.limit(20))

PULocationID,DOLocationID,trips
237,236,23708
236,237,20476
236,236,17062
237,237,16333
161,237,10822
237,161,9483
161,236,9229
239,238,8976
142,239,8675
239,142,8155


In [0]:
path = "/Volumes/workspace/default/nycyellowtaxi/taxi_zone_lookup.csv"
zone_lookup= spark.read.csv(path)
display(zone_lookup.limit(20))
print(zone_lookup.count())

_c0,_c1,_c2,_c3
LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone


266


In [0]:
lookup_path = "/Volumes/workspace/default/nycyellowtaxi/taxi_zone_lookup.csv"

try:
    zone_lookup = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(lookup_path)
        .select(
            F.col("LocationID").cast("int").alias("LocationID"),
            F.col("Borough").alias("borough"),
            F.col("Zone").alias("zone"),
            F.col("service_zone").alias("service_zone")
        )
    )

    pu = zone_lookup.select(
        F.col("LocationID").alias("PULocationID"),
        F.col("borough").alias("pickup_borough"),
        F.col("zone").alias("pickup_zone"),
        F.col("service_zone").alias("pickup_service_zone"),
    )

    do = zone_lookup.select(
        F.col("LocationID").alias("DOLocationID"),
        F.col("borough").alias("dropoff_borough"),
        F.col("zone").alias("dropoff_zone"),
        F.col("service_zone").alias("dropoff_service_zone"),
    )

    gold_zone_heatmap_named = (
        gold_zone_heatmap
        .join(pu, on="PULocationID", how="left")
        .join(do, on="DOLocationID", how="left")
        .orderBy(F.col("trips").desc())
    )

    display(gold_zone_heatmap_named.limit(20))
except Exception as e:
    print("Lookup enrichment skipped (could not read lookup file).")
    print("Error:", e)

DOLocationID,PULocationID,trips,pickup_borough,pickup_zone,pickup_service_zone,dropoff_borough,dropoff_zone,dropoff_service_zone
236,237,23708,Manhattan,Upper East Side South,Yellow Zone,Manhattan,Upper East Side North,Yellow Zone
237,236,20476,Manhattan,Upper East Side North,Yellow Zone,Manhattan,Upper East Side South,Yellow Zone
236,236,17062,Manhattan,Upper East Side North,Yellow Zone,Manhattan,Upper East Side North,Yellow Zone
237,237,16333,Manhattan,Upper East Side South,Yellow Zone,Manhattan,Upper East Side South,Yellow Zone
237,161,10822,Manhattan,Midtown Center,Yellow Zone,Manhattan,Upper East Side South,Yellow Zone
161,237,9483,Manhattan,Upper East Side South,Yellow Zone,Manhattan,Midtown Center,Yellow Zone
236,161,9229,Manhattan,Midtown Center,Yellow Zone,Manhattan,Upper East Side North,Yellow Zone
238,239,8976,Manhattan,Upper West Side South,Yellow Zone,Manhattan,Upper West Side North,Yellow Zone
239,142,8675,Manhattan,Lincoln Square East,Yellow Zone,Manhattan,Upper West Side South,Yellow Zone
142,239,8155,Manhattan,Upper West Side South,Yellow Zone,Manhattan,Lincoln Square East,Yellow Zone


### 3) gold_peak_hours
Trips by (hour, day_of_week).

In [0]:
gold_peak_hours = (
    gold_base
    .groupBy("pickup_day_of_week", "pickup_hour")
    .agg(F.count("*").alias("trips"))
    .orderBy(F.col("trips").desc())
)

display(gold_peak_hours.limit(10))

pickup_day_of_week,pickup_hour,trips
5,18,40082
5,17,39082
4,18,36249
6,17,35928
6,18,35795
4,17,35792
5,16,33962
5,19,33400
5,15,33229
5,21,32998


### 4) gold_payment_breakdown
Share of payment types (by trip count).

In [0]:
total_trips = gold_base.count()

gold_payment_breakdown = (
    gold_base
    .groupBy("payment_type")
    .agg(F.count("*").alias("trips"), F.sum("revenue").alias("revenue"))
    .withColumn("trip_share", F.col("trips") / F.lit(total_trips))
    .orderBy(F.col("trips").desc())
)

display(gold_payment_breakdown)

payment_type,trips,revenue,trip_share
1,2443133,6.860530718005762E7,0.8501008896171205
2,376088,8873132.48000089,0.1308617841821643
4,39068,1923273.9699999609,0.01359391468068323
3,15643,345010.7400000019,0.005443063564808226
5,1,0.0,3.4795522372998955E-7


### Sanity checks
Quick consistency checks (optional but interview-friendly).

In [0]:
print("silver_clean rows:", silver_nyc_yellow_taxi.count())
print("gold_base rows:", gold_base.count())
print("daily_kpis trips sum:", gold_daily_kpis.select(F.sum("trips")).collect()[0][0])
print("payment breakdown trips sum:", gold_payment_breakdown.select(F.sum("trips")).collect()[0][0])

silver_clean rows: 2873933
gold_base rows: 2873933
daily_kpis trips sum: 2873933
payment breakdown trips sum: 2873933
